# 06 · 评估指标模块（Metrics）功能演示

演示分类、特征、稳定性、金融风控四大类评估指标，与 sklearn/scipy 参考实现一致。

In [1]:
import warnings, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import hscredit

# 路径约定：从 notebooks/ 目录运行，数据在 ../examples，产物输出到 model_report/
DATA = os.path.join("..", "examples", "hscredit_yyp.xlsx")
if not os.path.exists(DATA):
    DATA = os.path.join("examples", "hscredit_yyp.xlsx")
OUT = "model_report"
os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(DATA)
df["放款时间"] = pd.to_datetime(df["放款时间"])
y = df["FPD"].astype(int)
NUM_FEATURES = ["珊瑚92", "青云24", "衡枢鉴真分老客版", "占信V3", "天创小额网贷分", "近六个月非银多头机构数"]
CAT_FEATURE = "商品类别"
print("数据形状:", df.shape)
print("坏样本率: {:.4f}".format(y.mean()))
df.head()

数据形状: (970, 18)
坏样本率: 0.1402


,客户编号,放款时间,放款金额,商品类别,MOB1,CURRENT_DPD,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,近六个月非银多头机构数,手机号近一个月非银多头机构数,身份证近一个月非银多头机构数,衡枢鉴真分老客版,FPD
0,1985945640026276096,2026-02-03,1399,礼包,0,0,NaN,NaN,NaN,656,NaN,NaN,630,51,15,15,0.0242,0
1,1985972188268592896,2026-02-04,1399,礼包,0,0,NaN,NaN,NaN,565,NaN,NaN,583,56,6,18,0.0492,0
2,1986034700861140992,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,708,NaN,NaN,764,68,17,20,0.0546,0
3,1986264852923760896,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,555,NaN,NaN,712,45,15,15,0.0899,0
4,1986265696509906944,2026-01-26,1399,礼包,0,0,NaN,NaN,NaN,581,NaN,NaN,641,67,32,32,0.0678,0


## 1. 准备评分与预测概率

In [2]:
from sklearn.model_selection import train_test_split
from hscredit.core.models import LightGBMRiskModel
from hscredit.core import metrics as M

Xn = df[NUM_FEATURES].fillna(0)
Xtr, Xte, ytr, yte = train_test_split(Xn, y, test_size=0.3, random_state=0, stratify=y)
m = LightGBMRiskModel(n_estimators=80).fit(Xtr, ytr)
p_tr, p_te = m.predict_proba(Xtr)[:,1], m.predict_proba(Xte)[:,1]
yte_a, ytr_a = yte.values, ytr.values
print('准备完成')

[1]	valid_0's binary_logloss: 0.399427
[2]	valid_0's binary_logloss: 0.396325
[3]	valid_0's binary_logloss: 0.394742
[4]	valid_0's binary_logloss: 0.395235
[5]	valid_0's binary_logloss: 0.395056
[6]	valid_0's binary_logloss: 0.396999
[7]	valid_0's binary_logloss: 0.399323
[8]	valid_0's binary_logloss: 0.400322
[9]	valid_0's binary_logloss: 0.402387
[10]	valid_0's binary_logloss: 0.40488
[11]	valid_0's binary_logloss: 0.405688
[12]	valid_0's binary_logloss: 0.407271
[13]	valid_0's binary_logloss: 0.409976
[14]	valid_0's binary_logloss: 0.41338
[15]	valid_0's binary_logloss: 0.417056
[16]	valid_0's binary_logloss: 0.420492
[17]	valid_0's binary_logloss: 0.424054
[18]	valid_0's binary_logloss: 0.430458
[19]	valid_0's binary_logloss: 0.430314
[20]	valid_0's binary_logloss: 0.435103
[21]	valid_0's binary_logloss: 0.435917
[22]	valid_0's binary_logloss: 0.441485
[23]	valid_0's binary_logloss: 0.441444
[24]	valid_0's binary_logloss: 0.439588
[25]	valid_0's binary_logloss: 0.440098
[26]	valid_

## 2. 分类指标：KS / AUC / Gini / 准确率族

In [3]:
pd.DataFrame([
    {'指标': 'KS', '值': round(M.ks(yte_a, p_te), 4)},
    {'指标': 'AUC', '值': round(M.auc(yte_a, p_te), 4)},
    {'指标': 'Gini', '值': round(M.gini(yte_a, p_te), 4)},
    {'指标': '准确率', '值': round(M.accuracy(yte_a, (p_te>0.5).astype(int)), 4)},
    {'指标': '精确率', '值': round(M.precision(yte_a, (p_te>0.5).astype(int)), 4)},
    {'指标': '召回率', '值': round(M.recall(yte_a, (p_te>0.5).astype(int)), 4)},
    {'指标': 'F1', '值': round(M.f1(yte_a, (p_te>0.5).astype(int)), 4)},
])

,指标,值
0,KS,0.2430
1,AUC,0.6482
2,Gini,0.2964
3,准确率,0.8625
4,精确率,0.5455
5,召回率,0.1463
6,F1,0.2308


## 3. KS 分桶表

In [4]:
M.ks_bucket(yte_a, p_te)

,桶编号,最小概率,最大概率,样本数,坏样本率,累积坏样本率,累积好样本率,KS贡献
0,0,0.0005,0.0024,30,0.0667,0.0667,0.9333,0.0632
1,1,0.0025,0.0052,29,0.0690,0.0678,0.9322,0.1224
2,2,0.0054,0.0097,27,0.0741,0.0698,0.9302,0.1737
3,3,0.0097,0.0172,31,0.1613,0.0940,0.9060,0.1557
4,4,0.0175,0.0271,28,0.1429,0.1034,0.8966,0.1541
5,5,0.0272,0.0391,30,0.0667,0.0971,0.9029,0.2174
6,6,0.0398,0.0683,29,0.1724,0.1078,0.8922,0.1914
7,7,0.0689,0.1046,28,0.1429,0.1121,0.8879,0.1899
8,8,0.1048,0.2579,29,0.1724,0.1188,0.8812,0.1639
9,9,0.2750,0.8086,30,0.3333,0.1409,0.8591,0.0000


## 4. 特征指标：IV / IV 明细表 / 卡方 / Cramér's V

In [5]:
print('单特征 IV:', round(M.iv(y, df['衡枢鉴真分老客版'].fillna(0)), 4))
print('卡方检验:', tuple(round(v,4) for v in M.chi2_test(df['商品类别'], y)))
print("Cramér's V:", round(M.cramers_v(df['商品类别'], y), 4))
M.iv_table(y, df['衡枢鉴真分老客版'].fillna(0))

单特征 IV: 0.2456
卡方检验: (4.4498, 0.4866)
Cramér's V: 0.0677


,分箱,分箱标签,样本总数,好样本数,坏样本数,样本占比,好样本占比,坏样本占比,坏样本率,分档WOE值,...,指标IV值,LIFT值,坏账改善,风险拒绝比,累积LIFT值,累积坏账改善,累计风险拒绝比,累积好样本数,累积坏样本数,分档KS值
0,0,"[-inf, 0.0370)",97,89,8,0.1000,0.1067,0.0588,0.0825,-0.5956,...,0.2456,0.5882,-0.0458,-0.4575,0.5882,-0.0458,-0.4575,89,8,0.0479
1,1,"[0.0370, 0.0478)",99,89,10,0.1021,0.1067,0.0735,0.1010,-0.3725,...,0.2456,0.7204,-0.0318,-0.3113,0.6550,-0.0874,-0.4323,178,18,0.0811
2,2,"[0.0478, 0.0602)",96,89,7,0.0990,0.1067,0.0515,0.0729,-0.7291,...,0.2456,0.5201,-0.0527,-0.5326,0.6106,-0.1677,-0.5570,267,25,0.1363
3,3,"[0.0602, 0.0719)",96,84,12,0.0990,0.1007,0.0882,0.1250,-0.1323,...,0.2456,0.8915,-0.0119,-0.1204,0.6801,-0.2132,-0.5331,351,37,0.1488
4,4,"[0.0719, 0.0838)",97,80,17,0.1000,0.0959,0.1250,0.1753,0.2648,...,0.2456,1.2500,0.0278,0.2778,0.7941,-0.2059,-0.4118,431,54,0.1197
5,5,"[0.0838, 0.0984)",97,86,11,0.1000,0.1031,0.0809,0.1134,-0.2429,...,0.2456,0.8088,-0.0212,-0.2124,0.7966,-0.3051,-0.5086,517,65,0.1420
6,6,"[0.0984, 0.1157)",97,88,9,0.1000,0.1055,0.0662,0.0928,-0.4665,...,0.2456,0.6618,-0.0376,-0.3758,0.7773,-0.5196,-0.7423,605,74,0.1813
7,7,"[0.1157, 0.1342)",98,83,15,0.1010,0.0995,0.1103,0.1531,0.1028,...,0.2456,1.0917,0.0103,0.1020,0.8170,-0.7369,-0.9199,688,89,0.1705
8,8,"[0.1342, 0.1679)",95,76,19,0.0979,0.0911,0.1397,0.2000,0.4273,...,0.2456,1.4265,0.0463,0.4728,0.8834,-1.0378,-1.1545,764,108,0.1219
9,9,"[0.1679, +inf)",98,70,28,0.1010,0.0839,0.2059,0.2857,0.8973,...,0.2456,2.0378,0.1166,1.1545,1.0000,1.0000,1.0000,834,136,0.0000


## 5. 稳定性指标：PSI / PSI 表 / CSI / 批量 PSI

In [6]:
print('评分 PSI:', round(M.psi(p_tr, p_te), 4), '->', M.psi_rating(M.psi(p_tr, p_te)))
display(M.psi_table(p_tr, p_te))
M.batch_psi(Xtr[NUM_FEATURES], Xte[NUM_FEATURES])

评分 PSI: 0.1857 -> 有轻微变化 (0.1 <= PSI < 0.25)


,分箱,期望样本数,实际样本数,期望占比,实际占比,PSI贡献
0,"[-inf, 0.0023)",72,27,0.1060,0.0928,0.0018
1,"[0.0023, 0.0056)",60,34,0.0884,0.1168,0.0080
2,"[0.0056, 0.0093)",74,24,0.1090,0.0825,0.0074
3,"[0.0093, 0.0151)",71,26,0.1046,0.0893,0.0024
4,"[0.0151, 0.0243)",71,26,0.1046,0.0893,0.0024
5,"[0.0243, 0.0366)",61,36,0.0898,0.1237,0.0108
6,"[0.0366, 0.0606)",70,27,0.1031,0.0928,0.0011
7,"[0.0606, 0.1058)",62,34,0.0913,0.1168,0.0063
8,"[0.1058, 0.4726)",53,45,0.0781,0.1546,0.0524
9,"[0.4726, +inf)",85,12,0.1252,0.0412,0.0932


,特征,PSI,评级
0,珊瑚92,0.0226,没有显著变化 (PSI < 0.1)
1,青云24,0.0547,没有显著变化 (PSI < 0.1)
2,衡枢鉴真分老客版,0.0896,没有显著变化 (PSI < 0.1)
3,占信V3,0.0404,没有显著变化 (PSI < 0.1)
4,天创小额网贷分,0.0212,没有显著变化 (PSI < 0.1)
5,近六个月非银多头机构数,0.0256,没有显著变化 (PSI < 0.1)


## 6. 金融风控指标：Lift / Lift 表 / 坏率 / 评分分布

In [7]:
print('整体 Lift:', round(M.lift(yte_a, p_te), 4))
print('Top 10% Lift:', round(M.lift_at(yte_a, p_te, 0.1), 4))
display(M.lift_table(yte_a, p_te))
M.score_stats(p_te*1000)

整体 Lift: 3.8714
Top 10% Lift: 2.4474


,分箱,最小概率,最大概率,样本数,好样本数,坏样本数,坏样本率,样本占比,Lift值,坏账改善,累积Lift值,累积坏账改善
0,1,0.0005,0.0024,30,28,2,0.0667,0.1031,0.4732,-0.0606,0.4732,-0.0606
1,2,0.0025,0.0052,29,27,2,0.0690,0.0997,0.4895,-0.0565,0.4812,-0.1319
2,3,0.0054,0.0097,27,25,2,0.0741,0.0928,0.5257,-0.0485,0.4952,-0.2118
3,4,0.0097,0.0172,31,26,5,0.1613,0.1065,1.1448,0.0173,0.6673,-0.2237
4,5,0.0175,0.0271,28,24,4,0.1429,0.0962,1.0139,0.0015,0.7342,-0.2639
5,6,0.0272,0.0391,30,28,2,0.0667,0.1031,0.4732,-0.0606,0.6895,-0.4685
6,7,0.0398,0.0683,29,24,5,0.1724,0.0997,1.2237,0.0248,0.7654,-0.5500
7,8,0.0689,0.1046,28,24,4,0.1429,0.0962,1.0139,0.0015,0.7954,-0.8045
8,9,0.1048,0.2579,29,24,5,0.1724,0.0997,1.2237,0.0248,0.8430,-1.3659
9,10,0.2750,0.8086,30,20,10,0.3333,0.1031,2.3659,0.1570,1.0000,1.0000


{'样本数': 291,
 '缺失数': 0,
 '缺失率': 0.0,
 '均值': 84.82052054098068,
 '标准差': 143.83751358347763,
 '最小值': 0.461942054842104,
 '最大值': 808.6488501647285,
 '中位数': 27.2267718668365,
 '分位数_25': 7.054218896293451,
 '分位数_75': 84.4230388831509}

## 7. 指标汇总导出

In [8]:
M.ks_bucket(yte_a, p_te).to_excel(f"{OUT}/06_metrics_ks_bucket.xlsx", index=False)
M.lift_table(yte_a, p_te).to_excel(f"{OUT}/06_metrics_lift_table.xlsx", index=False)
print('已保存 KS 分桶表与 Lift 表')

已保存 KS 分桶表与 Lift 表
